# Three extractors, one notebook — the extraction-robustness comparison

Run **three** code extractors on the *same* model completions and compare, to answer the key
question for the paper: *is the "+9-pt politeness tax" a real effect, or an artifact of naive
extraction?*

- **(a) naive** — the pre-fix extractor that produced the +9.3 (first fenced block, else the whole
  fence-stripped reply **with trailing prose**).
- **(b) ours** — the robust extractor from `score_lcb.py` (trim-to-compilable, keep helpers,
  shape-agnostic).
- **(c) EvalPlus `sanitize`** — the independent, widely-used reference tool (tree-sitter AST;
  longest-valid-substring; reachability from `entrypoint`).

**The headline metric is compile-rate by condition** — the share of completions whose extracted
code parses *and* defines the target. It needs **only `lcb_v3_responses.json`** (no 400MB tests
file). If the naive extractor's compile-rate is **lower for the wordier conditions** (polite/
multilingual) and both robust extractors are **~100% across all conditions**, that *is* the
artifact. (Optional later cells add the full pass-rate comparison if you upload `lcb_tests.jsonl`.)

## 0 · Install tree-sitter (for the vendored `sanitize`)
We run EvalPlus's `sanitize` **vendored** (byte-identical functions, cell 4) to avoid the full
`evalplus` package's dataset deps — it needs only `tree_sitter` + the Python grammar. Errors are
shown (not hidden), so if this fails you'll see why.

In [ ]:
!pip install tree_sitter tree_sitter_python

## 1 · Upload data
`lcb_v3_responses.json` is **required**. `lcb_problems.jsonl` (difficulty) and `lcb_tests.jsonl`
(for the optional pass-rate cells) are optional.

In [ ]:
import json
try:
    from google.colab import files
    up = files.upload()
except Exception:
    up = {n: open(n, "rb").read() for n in
          ("lcb_v3_responses.json", "lcb_problems.jsonl", "lcb_tests.jsonl") if __import__("os").path.exists(n)}
PROBS, RESP_LIST, TESTS = {}, [], {}
for name, data in up.items():
    text = data.decode("utf-8", errors="replace")
    if text.lstrip().startswith("["):
        RESP_LIST = json.loads(text); print(f"[responses] {len(RESP_LIST)} completions")
    else:
        rows = [json.loads(l) for l in text.splitlines() if l.strip()]
        if rows and ("public_tests" in rows[0] or "private_tests" in rows[0]):
            TESTS = {d["task_id"]: d for d in rows}; print(f"[tests] {len(TESTS)}")
        else:
            PROBS = {d["task_id"]: d for d in rows}; print(f"[problems] {len(PROBS)}")
assert RESP_LIST, "lcb_v3_responses.json is required."
CONDS = sorted({r["level"] for r in RESP_LIST})
SANITIZE_OK = False   # set True by cell 4 on success
print("conditions:", CONDS)

## 1b · (Optional) Test-count diagnostic — which problems have the most tests?
Needs `lcb_tests.jsonl`. Shows the distribution and exactly which problems exceed `MAX_TESTS`
(the only ones where the cap samples instead of running every test).

In [ ]:
if not TESTS:
    print("upload lcb_tests.jsonl to see test counts.")
else:
    import statistics as st
    counts = sorted(((len(t.get("public_tests", [])) + len(t.get("private_tests", [])), tid)
                     for tid, t in TESTS.items()), reverse=True)
    ns = [c for c, _ in counts]
    print(f"{len(counts)} problems | tests/problem: min {min(ns)} / median {int(st.median(ns))} "
          f"/ mean {st.mean(ns):.1f} / max {max(ns)}")
    print("\ntop 10 by test count:")
    for n, tid in counts[:10]:
        print(f"  {tid:10s} {n} tests")
    CAP = 60
    capped = [(n, tid) for n, tid in counts if n > CAP]
    print(f"\nproblems with >{CAP} tests (sampled when MAX_TESTS={CAP}): "
          f"{len(capped)} ({100*len(capped)/len(counts):.0f}% of problems)")
    for n, tid in capped[:20]:
        print(f"  {tid:10s} {n} tests -> sampled to {CAP}")

## 2 · Define OUR extractor + scorer (inline, from `score_lcb.py`)

In [ ]:
# --- OUR extractor + scorer, copied verbatim from score_lcb.py ---
import ast, json, re, time, multiprocessing, inspect

_IMPORTS = 'from typing import *\nimport collections, math, heapq, bisect, itertools, functools, re\nfrom collections import *\nfrom math import *\nfrom functools import *\n'
PER_TEST_TIMEOUT = 6
PRIVATE_ONLY = False

def parse_lit(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except Exception:
            continue
    return s  # leave as raw string


def parse_input(inp):
    """LCB functional input = one literal per line -> list of args."""
    if isinstance(inp, list):
        return [parse_lit(x) for x in inp]
    return [parse_lit(line) for line in str(inp).split("\n") if line.strip() != ""]


def _trim_to_compilable(code):
    """Drop trailing lines until the source parses. Conversational replies put a
    prose paragraph AFTER the code ('This checks every adjacent pair...'); with no
    fences to delimit it, that prose used to be exec'd and threw SyntaxError,
    failing correct code. The polite/detailed framing elicits more explain-after-
    code, so this bug penalized it hardest — a scoring artifact, not correctness.
    We keep the largest leading prefix that is valid Python and still has the
    class/method."""
    lines = code.split("\n")
    while lines:
        src = "\n".join(lines)
        try:
            ast.parse(src)
            return src
        except SyntaxError:
            lines.pop()
    return None


def extract_solution(completion, entry):
    """Pull runnable code that defines `class Solution` (or the method), robust to
    trailing prose, leading chatter, AND helper definitions (Fenwick/DSU/Segments…)
    the model defines BEFORE `class Solution`. Returns the first candidate that
    compiles and still defines the target. Candidates that KEEP the helpers are
    tried first, so we don't hand exec() a Solution that references an undefined
    helper we chopped off."""
    if not completion:
        return None
    text = "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)
    code_blocks = [b for b in blocks if re.search(r"(?m)^\s*(?:class|def|import|from|@)\s", b)]

    cands = []
    # (B) from the FIRST top-level code construct to the end — keeps helper
    #     classes/functions defined above `class Solution` (fixes the drop bug).
    m = re.search(r"(?m)^(?:class|def|import|from|@)\s", text)
    if m:
        cands.append(text[m.start():])
    # (A) all fenced code blocks concatenated — helper and Solution in separate blocks.
    if code_blocks:
        cands.append("\n\n".join(code_blocks))
    # (C) a single block that names the target (old high-confidence path).
    for b in blocks:
        if "class Solution" in b or f"def {entry}" in b:
            cands.append(b)
    # (D) old anchor slice, last resort.
    for anchor in ("class Solution", f"def {entry}"):
        i = text.find(anchor)
        if i != -1:
            cands.append(text[i:])

    for c in cands:
        t = _trim_to_compilable(c)
        if t and ("class Solution" in t or f"def {entry}" in t):
            return t
    return cands[0] if cands else None


def resolve_callable(ns, entry):
    """Find the `entry` implementation regardless of how the model SHAPED its code.

    LCB expects `Solution().<entry>(*args)`. But a correct solution is sometimes
    delivered in a different container — a bare module-level function, a method on a
    RENAMED top-level class, or a method on a class nested one level inside another.
    Those are code *smells*, not wrong answers, yet the old resolver (hardcoded
    `ns['Solution']`) scored them FAIL. We look past the shape and bind to the actual
    implementation. Returns a callable (bound, if a method) or None.

    Order: (1) a bare module-level function named `entry`; else (2) a class exposing
    a callable `entry`, preferring one literally named `Solution`, searching both
    top-level classes and classes nested one level inside another class."""
    import inspect
    f = ns.get(entry)
    if callable(f) and not inspect.isclass(f):
        return f                                   # bare function shape
    classes = [v for v in ns.values() if inspect.isclass(v)]
    nested = [v for c in classes for v in vars(c).values() if inspect.isclass(v)]
    cands = [c for c in (classes + nested) if callable(getattr(c, entry, None))]
    if not cands:
        return None
    cands.sort(key=lambda c: c.__name__ != "Solution")   # prefer the expected name
    try:
        inst = cands[0]()
    except Exception:
        return None
    return getattr(inst, entry, None)


def eq(a, b):
    if isinstance(a, float) or isinstance(b, float):
        try:
            return abs(float(a) - float(b)) < 1e-6
        except Exception:
            return False
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(eq(x, y) for x, y in zip(a, b))
    return a == b


def _worker(code, entry, tests, q):
    """Run tests in order, STREAMING one bool per test into q, and stop at the
    first failure (early exit). Streaming lets passes() enforce a PER-TEST timeout
    instead of one budget for the whole suite — a correct-but-slow solution on a
    many-test problem must not be failed just because the SUM of test times exceeds
    a single cap (LCB, like any judge, limits each test, not the total)."""
    ns = {}
    try:
        exec(_IMPORTS + code, ns)
    except Exception:
        q.append(False); return
    fn = resolve_callable(ns, entry)               # shape-agnostic (class/bare/renamed/nested)
    for t in tests:
        try:
            args = parse_input(t["input"])
            expected = parse_lit(t["output"])
            ok = (fn is not None) and eq(fn(*args), expected)
        except Exception:
            ok = False
        q.append(ok)
        if not ok:
            return                      # early exit: one failure fails the problem


def passes(code, entry, tests):
    """True iff EVERY test passes, each within PER_TEST_TIMEOUT seconds. A wrong
    answer fails fast; a test that hangs (infinite loop / TLE) fails on its own
    timeout without killing the tests that already passed."""
    if not code or not tests:
        return False
    mgr = multiprocessing.Manager(); q = mgr.list()
    p = multiprocessing.Process(target=_worker, args=(code, entry, tests, q))
    p.start()
    seen = 0; last_progress = time.time()
    while p.is_alive():
        n = len(q)
        if n > seen:
            if q[n - 1] is False:                 # newest test failed -> stop
                p.kill(); p.join(2); return False
            seen = n; last_progress = time.time()
        elif time.time() - last_progress > PER_TEST_TIMEOUT:   # hung on current test
            p.kill(); p.join(2); return False
        time.sleep(0.02)
    res = list(q)
    return len(res) == len(tests) and all(res)    # all tests present and passed


def sample_tests(rec, k):
    pub = [] if PRIVATE_ONLY else rec.get("public_tests", [])
    priv = rec.get("private_tests", [])
    if len(pub) + len(priv) <= k:
        return pub + priv
    need = max(0, k - len(pub))
    stride = max(1, len(priv) // need) if need else 1
    return pub + (priv[::stride][:need] if need else [])


extract_ours = extract_solution
print("OUR extractor + scorer defined (extract_ours).")

## 3 · Define the NAIVE extractor (the one that produced +9.3)

In [ ]:
def extract_naive(completion, entry):
    """Pre-fix extractor: first fenced block naming the target, else the whole
    fence-stripped completion -- trailing prose and all (-> SyntaxError on correct code)."""
    if not completion:
        return None
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)
    for b in blocks:
        if "class Solution" in b or f"def {entry}" in b:
            return b
    if "class Solution" in completion or f"def {entry}" in completion:
        return "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
    return None

print("naive extractor defined (extract_naive).")

## 4 · Define EvalPlus `sanitize` (vendored — byte-identical, needs only tree-sitter)
If the imports here fail, run cell 0 first. This is EvalPlus's real `sanitize` algorithm
(`code_extract` longest-valid-substring + reachability-based `extract_target_code_or_empty`).

In [ ]:
# Vendored EvalPlus sanitize() -- the byte-identical functions from evalplus/sanitize.py,
# stripped of the evalplus.data/tqdm/fire CLI deps so it needs ONLY tree_sitter + the
# Python grammar. This IS EvalPlus's sanitize algorithm (code_extract + reachability).
import re, ast
from typing import Dict, Generator, List, Optional, Set, Tuple
import tree_sitter_python
from tree_sitter import Language, Node, Parser

def syntax_check(code, verbose=False):
    try:
        ast.parse(code)
        return True
    except (SyntaxError, MemoryError):
        return False

CLASS_TYPE = "class_definition"
FUNCTION_TYPE = "function_definition"
IMPORT_TYPE = ["import_statement", "import_from_statement"]
IDENTIFIER_TYPE = "identifier"
ATTRIBUTE_TYPE = "attribute"
RETURN_TYPE = "return_statement"
EXPRESSION_TYPE = "expression_statement"
ASSIGNMENT_TYPE = "assignment"


def code_extract(text: str) -> str:
    # Remove ANSI escape sequences, which was caused by ollama (small qwen3 models, but may be others as well)
    ansi_escape = re.compile(r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])")
    text = ansi_escape.sub("", text)

    lines = text.split("\n")
    longest_line_pair = (0, 0)
    longest_so_far = 0

    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            current_lines = "\n".join(lines[i : j + 1])
            if syntax_check(current_lines):
                current_length = sum(1 for line in lines[i : j + 1] if line.strip())
                if current_length > longest_so_far:
                    longest_so_far = current_length
                    longest_line_pair = (i, j)

    return "\n".join(lines[longest_line_pair[0] : longest_line_pair[1] + 1])


def get_deps(nodes: List[Tuple[str, Node]]) -> Dict[str, Set[str]]:
    def dfs_get_deps(node: Node, deps: Set[str]) -> None:
        for child in node.children:
            if child.type == IDENTIFIER_TYPE:
                deps.add(child.text.decode("utf8"))
            else:
                dfs_get_deps(child, deps)

    name2deps = {}
    for name, node in nodes:
        deps = set()
        dfs_get_deps(node, deps)
        name2deps[name] = deps
    return name2deps


def get_function_dependency(entrypoint: str, call_graph: Dict[str, str]) -> Set[str]:
    queue = [entrypoint]
    visited = {entrypoint}
    while queue:
        current = queue.pop(0)
        if current not in call_graph:
            continue
        for neighbour in call_graph[current]:
            if not (neighbour in visited):
                visited.add(neighbour)
                queue.append(neighbour)
    return visited


def get_definition_name(node: Node) -> str:
    for child in node.children:
        if child.type == IDENTIFIER_TYPE:
            return child.text.decode("utf8")


def traverse_tree(node: Node) -> Generator[Node, None, None]:
    cursor = node.walk()
    depth = 0

    visited_children = False
    while True:
        if not visited_children:
            yield cursor.node
            if not cursor.goto_first_child():
                depth += 1
                visited_children = True
        elif cursor.goto_next_sibling():
            visited_children = False
        elif not cursor.goto_parent() or depth == 0:
            break
        else:
            depth -= 1


def has_return_statement(node: Node) -> bool:
    traverse_nodes = traverse_tree(node)
    for node in traverse_nodes:
        if node.type == RETURN_TYPE:
            return True
    return False


def extract_target_code_or_empty(code: str, entrypoint: Optional[str] = None) -> str:
    code = code_extract(code)
    code_bytes = bytes(code, "utf8")
    parser = Parser(Language(tree_sitter_python.language()))
    tree = parser.parse(code_bytes)
    class_names = set()
    function_names = set()
    variable_names = set()

    root_node = tree.root_node
    import_nodes = []
    definition_nodes = []

    for child in root_node.children:
        if child.type in IMPORT_TYPE:
            import_nodes.append(child)
        elif child.type == CLASS_TYPE:
            name = get_definition_name(child)
            if not (
                name in class_names or name in variable_names or name in function_names
            ):
                definition_nodes.append((name, child))
                class_names.add(name)
        elif child.type == FUNCTION_TYPE:
            name = get_definition_name(child)
            if not (
                name in function_names or name in variable_names or name in class_names
            ) and has_return_statement(child):
                definition_nodes.append((name, child))
                function_names.add(get_definition_name(child))
        elif (
            child.type == EXPRESSION_TYPE and child.children[0].type == ASSIGNMENT_TYPE
        ):
            subchild = child.children[0]
            name = get_definition_name(subchild)
            if not (
                name in variable_names or name in function_names or name in class_names
            ):
                definition_nodes.append((name, subchild))
                variable_names.add(name)

    if entrypoint:
        name2deps = get_deps(definition_nodes)
        reacheable = get_function_dependency(entrypoint, name2deps)

    sanitized_output = b""

    for node in import_nodes:
        sanitized_output += code_bytes[node.start_byte : node.end_byte] + b"\n"

    for pair in definition_nodes:
        name, node = pair
        if entrypoint and not (name in reacheable):
            continue
        sanitized_output += code_bytes[node.start_byte : node.end_byte] + b"\n"
    return sanitized_output[:-1].decode("utf8")


def sanitize(code: str, entrypoint: Optional[str] = None) -> str:
    sanitized_code = extract_target_code_or_empty(code, entrypoint).strip()
    if not sanitized_code:
        return code_extract(code)
    return sanitized_code


def extract_sanitize(completion, entry):
    try:
        return sanitize(completion or '', entrypoint=entry)
    except Exception as e:
        print('sanitize call error:', type(e).__name__, e)
        return None
SANITIZE_OK = True
print('vendored EvalPlus sanitize ready (extract_sanitize).')


## (d) LCB official `extract_code` — the fourth extractor

LiveCodeBench's own extraction function (`lcb_runner/utils/extraction_utils.py`), verbatim (the generic/API-model branch). It is **fence-only**: it returns the text between the last two ` ``` ` lines and **returns empty when a reply has no fences**. LCB's own prompts instruct fenced output; our system prompt says *'plain Python, no fences'*, so on our replies it returns nothing ~77% of the time, and where it does fire it tracks how often a model happened to use fences — which *rises with politeness*, producing a **reversed** 'politeness bonus'. This is not a flaw in LCB: it shows an extractor's correctness is contingent on the prompt's output convention, and a mismatch silently corrupts results (here, in the opposite direction to naive).

In [ ]:
def extract_lcb(completion, entry):
    """LiveCodeBench official extract_code (lcb_runner/utils/extraction_utils.py), generic/API
    branch, VERBATIM: text between the LAST TWO ``` fence lines; "" if < 2 fence lines. Fence-only,
    so 'no fences' replies yield nothing. `entry` is unused (the official fn does not anchor on the
    target); kept for a uniform signature."""
    if completion is None:
        return ""
    outputlines = completion.split("\n")
    indexlines = [i for i, line in enumerate(outputlines) if "```" in line]
    if len(indexlines) < 2:
        return ""
    return "\n".join(outputlines[indexlines[-2] + 1 : indexlines[-1]])

print("LCB official extractor defined (extract_lcb).")

## 5 · Compile-rate-by-condition (the headline metric — no tests needed)
A completion "compiles" if its extracted code parses **and** still defines the target.

In [ ]:
import ast
from collections import defaultdict

def compile_ok(code, entry):
    if not code:
        return False
    try:
        ast.parse(code)
    except SyntaxError:
        return False
    return ("class Solution" in code) or (f"def {entry}" in code)

def compile_rate_by_condition(extractor, records=None):
    records = records if records is not None else RESP_LIST
    agg = defaultdict(lambda: [0, 0])
    for r in records:
        entry = r["entry_point"]
        try:
            code = extractor(r.get("completion"), entry)
        except Exception:
            code = None
        agg[r["level"]][0] += compile_ok(code, entry)
        agg[r["level"]][1] += 1
    return {k: round(100 * v[0] / v[1], 1) for k, v in sorted(agg.items())}

print("compile_rate_by_condition() ready.")

### 5a · NAIVE — compile-rate by condition

In [ ]:
naive_cr = compile_rate_by_condition(extract_naive)
for k, v in naive_cr.items():
    print(f"  {k:22s} {v}%")

### 5b · OURS — compile-rate by condition

In [ ]:
ours_cr = compile_rate_by_condition(extract_ours)
for k, v in ours_cr.items():
    print(f"  {k:22s} {v}%")

### 5c · EvalPlus `sanitize` — compile-rate by condition

In [ ]:
if SANITIZE_OK:
    sanitize_cr = compile_rate_by_condition(extract_sanitize)
    for k, v in sanitize_cr.items():
        print(f"  {k:22s} {v}%")
else:
    sanitize_cr = {k: None for k in CONDS}
    print("sanitize unavailable -- run cell 0 and re-run cells 4 and 5c.")

In [ ]:
lcb_cr = compile_rate_by_condition(extract_lcb)
for k, v in lcb_cr.items():
    print(f"  {k:22s} {v}%")

## 6 · Side-by-side comparison (the money table)
If the effect is an extraction artifact, the **naive** column is lower for wordier conditions,
while **ours** and **sanitize** are ~100% across the board.

In [ ]:
w = 22
cols = [("naive", naive_cr), ("ours", ours_cr), ("sanitize", sanitize_cr), ("lcb", lcb_cr)]
print(f"{'condition':{w}}" + "".join(f"{n:>10}" for n, _ in cols))
print("-" * (w + 10 * len(cols)))
for c in CONDS:
    row = f"{c:{w}}"
    for n, cr in cols:
        v = cr.get(c)
        row += (f"{v:>9}%" if v is not None else f"{'n/a':>10}")
    print(row)
print("\nReading: naive penalizes wordier conditions (+tax); ours + sanitize ~flat (null);")
print("LCB is fence-only: it returns empty on replies with <2 fences (our no-fence prompt),")
print("so it fires on a minority of replies and REVERSES (detailed > terse) because fenced")
print("formatting rises with politeness. The pass-rate + McNemar below are the authoritative test.")

### Reading the table — focus on ASYMMETRY, not absolute level
The artifact is a **condition asymmetry** (detailed/polite extracted *worse* than terse). Look at
the **spread** (max − min) within each column:
- **naive:** large spread, detailed is the *lowest* → this is what manufactures the +9.3.
- **ours:** ~0 spread (flat ~100%).
- **sanitize:** ~flat too (small spread, detailed no longer lowest) — **the asymmetry is gone.**

Note sanitize's *absolute* rate is lower than ours: it is HumanEval-oriented (its reachability
targets *top-level* functions, but LCB's entry is a *method* on `class Solution`), so it mostly
falls back to its longest-valid-substring extractor. Different algorithm, lower absolute level —
but it **still removes the asymmetry**, which is the robustness point (two independent robust
extractors agree the penalty is naive-extraction-specific).

## 7 · (Optional) Full PASS-rate by condition — needs `lcb_tests.jsonl`
This reproduces the actual +9.3 -> null by running the graded tests under each extractor. Heavier
(runs tests). Defaults to the capable models (chatgpt, claude) where the effect was reported.

In [ ]:
# ===== PASS-RATE + paired McNemar across the three extractors (needs lcb_tests.jsonl) =====
# Reproduces the headline: the terse-vs-detailed gap (+9.3, p<0.01 under naive) and whether it
# survives under ours / sanitize. Uses the SAME scorer as the paper (passes(), per-test timeout).
import multiprocessing as _mp
try: _mp.set_start_method("fork", force=True)   # Linux/Colab; harmless if already set
except Exception: pass
from math import comb

MODELS    = {"chatgpt", "claude"}   # capable slice; set to None for ALL three models
MAX_TESTS = 60                      # tests sampled per problem (matches score_lcb default)
SAMPLE_N  = None                    # e.g. 40 = quick smoke test on 40 random problems; None = all

def _records():
    recs = [r for r in RESP_LIST if (MODELS is None or r["model"] in MODELS)]
    if SAMPLE_N:
        import random; random.seed(0)
        ids = sorted({r["task_id"] for r in recs})
        keep = set(random.sample(ids, min(SAMPLE_N, len(ids))))
        recs = [r for r in recs if r["task_id"] in keep]
    return recs

def score_all(extractor, recs):
    out = {}
    for i, r in enumerate(recs):
        rec = TESTS.get(r["task_id"])
        if not rec: continue
        code = extractor(r.get("completion"), r["entry_point"])
        out[(r["task_id"], r["level"], r["model"])] = bool(
            passes(code, r["entry_point"], sample_tests(rec, MAX_TESTS)))
        if (i + 1) % 200 == 0: print(f"    ...{i+1}/{len(recs)}", flush=True)
    return out

def rate_by_cond(scored):
    agg = defaultdict(lambda: [0, 0])
    for (t, l, m), p in scored.items(): agg[l][0] += p; agg[l][1] += 1
    return {k: round(100 * v[0] / v[1], 1) for k, v in sorted(agg.items())}

def mcnemar(scored, a="agentic_terse", b="webchat_detailed"):
    keys = {(t, m) for (t, l, m) in scored}
    a_only = b_only = 0
    for (t, m) in keys:
        pa = scored.get((t, a, m)); pb = scored.get((t, b, m))
        if pa is None or pb is None: continue
        if pa and not pb: a_only += 1
        elif pb and not pa: b_only += 1
    n = a_only + b_only
    if n == 0:
        p = 1.0
    elif n <= 1000:
        tail = sum(comb(n, k) for k in range(0, min(a_only, b_only) + 1)); p = min(1.0, 2 * tail / (2 ** n))
    else:
        import math; z = abs(a_only - b_only) / math.sqrt(n); p = math.erfc(z / math.sqrt(2))
    return {f"{a}_only": a_only, f"{b}_only": b_only, "p": round(p, 4)}

if not TESTS:
    print("Upload lcb_tests.jsonl (Drive recommended, ~400MB) to run the pass-rate comparison.")
else:
    recs = _records()
    print(f"scoring {len(recs)} (problem x condition x model) records x each extractor")
    print(f"MODELS={MODELS}  MAX_TESTS={MAX_TESTS}  SAMPLE_N={SAMPLE_N}")
    print("NOTE: this runs the graded tests -> SLOW (several minutes). Set SAMPLE_N=40 for a quick check.\n")
    extractors = [("naive", extract_naive), ("ours", extract_ours)] + (
        [("sanitize", extract_sanitize)] if SANITIZE_OK else []) + [("lcb", extract_lcb)]
    pr = {}
    for name, extr in extractors:
        print(f"scoring {name} ...", flush=True)
        s = score_all(extr, recs); pr[name] = rate_by_cond(s)
        mc = mcnemar(s)
        tv = pr[name].get("agentic_terse"); dv = pr[name].get("webchat_detailed")
        dd = round(tv - dv, 1) if (tv is not None and dv is not None) else None
        print(f"  pass-rate by condition: {pr[name]}")
        print(f"  terse - detailed = {dd} pts   |   McNemar {mc}\n")
    # money table
    print("=== pass-rate by condition (the +9.3 -> null check) ===")
    w = 22; cols = [n for n, _ in extractors]
    print(f"{'condition':{w}}" + "".join(f"{c:>10}" for c in cols))
    print("-" * (w + 10 * len(cols)))
    for cond in CONDS:
        print(f"{cond:{w}}" + "".join(f"{pr[c].get(cond, '-'):>9}%" for c in cols))
    print("\nterse-vs-detailed: ~+16 pts (naive, artifact) · ~0 (ours & sanitize, null) · reversed/near-zero")
    print("(lcb, fence-only on our no-fence replies). Same completions, four extractors, three outcomes.")

---
**What to look for.** Compile-rate (cell 6): naive lower for polite/multilingual, ours + sanitize
~100%. Pass-rate (cell 7): the terse-vs-polite gap is large under naive and ~0 under both robust
extractors. Effect vanishing under **both** ours **and** the independent EvalPlus tool is the
robustness result that answers "is your extractor the real artifact?" 